# Verify `index_hosted_b16.html` end-to-end: real network fetch, real in-browser inference

`index_hosted_b16.html` (this directory's `web_demo/`) is meant to be self-sufficient: no local
`npm install`, no local `.tflite` file -- `@litertjs/core` loads from a CDN and the model is
fetched at runtime from [`huggingface.co/1kaiser/pointdit-litert`](https://huggingface.co/1kaiser/pointdit-litert).
This notebook is the reproducible version of that check (matching this project's own
"verify by running, not by reading the code" discipline throughout) -- it serves the page
locally (never as a bare `file://` -- same CORS reason as every other GLB/HTML page in this
project), drives it with a real headless Chrome via Playwright, and asserts on the actual
printed result rather than just "the subprocess exited 0".

## 1. Serve the demo directory locally

Never open as `file://` -- a `null`-origin page can't `fetch()` its own sibling assets
(`shapes.json`, the `.bin` tensors), the same CORS gap documented for the GLB/model-viewer
pages elsewhere in this project.

In [1]:
web_demo_dir = "tools/litert/web_demo"
demo_html = "index_hosted_b16.html"
node_bin = "/home/kaiser/.conda/envs/node20/bin/node"
timeout_s = 150

In [2]:
import functools
import http.server
import json
import subprocess
import threading
from pathlib import Path

handler = functools.partial(http.server.SimpleHTTPRequestHandler, directory=web_demo_dir)
httpd = http.server.ThreadingHTTPServer(("127.0.0.1", 0), handler)
threading.Thread(target=httpd.serve_forever, daemon=True).start()
port = httpd.server_address[1]
url = f"http://127.0.0.1:{port}/{demo_html}"
print(f"serving {web_demo_dir} at {url}")

serving tools/litert/web_demo at http://127.0.0.1:42977/index_hosted_b16.html


## 2. Drive it with real headless Chrome and capture the real result

`run_demo_hosted_b16.js` prints a single `RESULT: {...}` JSON line once
`window.__demoDone` is set -- the same page-side completion signal every browser demo driver in
this project uses. This is a real network-bound run (downloads the actual 139.7MB model from
HF over the network), not a cached/mocked one -- expect ~30-60s just for the download, before
any inference happens.

In [3]:
result = subprocess.run(
    [node_bin, f"{web_demo_dir}/run_demo_hosted_b16.js", url],
    capture_output=True, text=True, timeout=timeout_s,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr[-2000:])
httpd.shutdown()
assert result.returncode == 0, "run_demo_hosted_b16.js exited non-zero"

127.0.0.1 - - [27/Aug/2026 07:48:03] "GET /index_hosted_b16.html HTTP/1.1" 200 -


127.0.0.1 - - [27/Aug/2026 07:48:03] "GET /assets_b16/shapes.json HTTP/1.1" 200 -
127.0.0.1 - - [27/Aug/2026 07:48:03] code 404, message File not found
127.0.0.1 - - [27/Aug/2026 07:48:03] "GET /favicon.ico HTTP/1.1" 404 -


127.0.0.1 - - [27/Aug/2026 07:48:14] "GET /assets_b16/labels.bin HTTP/1.1" 200 -
127.0.0.1 - - [27/Aug/2026 07:48:14] "GET /assets_b16/cached_y_emb.bin HTTP/1.1" 200 -
127.0.0.1 - - [27/Aug/2026 07:48:14] "GET /assets_b16/t.bin HTTP/1.1" 200 -
127.0.0.1 - - [27/Aug/2026 07:48:14] "GET /assets_b16/z.bin HTTP/1.1" 200 -
127.0.0.1 - - [27/Aug/2026 07:48:14] "GET /assets_b16/ref_output.bin HTTP/1.1" 200 -


[console] shapes: {"z":[1,3,512,512],"t":[1],"labels":[1,3,512,512],"cached_y_emb":[1,1024,3072],"ref_output":[1,3,512,512],"image_name":"IMG_7261.png"}
[console] Failed to load resource: the server responded with a status of 404 (File not found)
[console] A valid external Instance reference no longer exists.
[console] WebGPU adapter: vendor=nvidia architecture=blackwell device=0x2c34
[console] loading LiteRT wasm runtime (CDN)...
[console] INFO: [environment.cc:36] Creating LiteRT environment with options
[console] WARNING: [npu_registry.cc:34] NPU accelerator could not be loaded and registered: kLiteRtStatusErrorInvalidArgument.
[console] INFO: [accelerator_registry.cc:54] RegisterAccelerator: ptr=0xcaa90, name=WebNN
[console] INFO: [webnn_registry.cc:35] Statically linked WebNN accelerator registered.
[console] INFO: [accelerator_registry.cc:54] RegisterAccelerator: ptr=0xcab30, name=GPU WebGPU
[console] INFO: [tensor_buffer_registry.cc:42] Same custom tensor buffer handler has alre

## 3. Verify the real numbers, not just that it printed something

A page that fetched nothing and silently no-op'd would still exit 0 -- the actual assertion is
on `RESULT`'s real fields: the model genuinely downloaded (a non-trivial size), inference
genuinely ran (a non-zero latency), and its accuracy matches the already-established
weight-only int8 B/16 number (0.007-0.008 max abs diff vs. the PyTorch-GPU reference; see
`run_litert_inference.py` and the main README's LiteRT benchmark table) rather than some
unrelated value that would indicate a silent fallback or a stale/wrong model file.

In [4]:
result_line = next(line for line in result.stdout.splitlines() if line.startswith("RESULT:"))
demo_result = json.loads(result_line[len("RESULT:"):].strip())
print(demo_result)

assert demo_result["latencyMs"] > 0, "zero-latency result -- inference likely didn't really run"
assert 0.005 < demo_result["maxDiff"] < 0.02, (
    f"max abs diff {demo_result['maxDiff']} is outside the expected weight-only int8 B/16 range "
    f"(0.007-0.008 previously measured) -- something about the fetched model or inference path "
    f"may be wrong, not just noisier than usual"
)
print(f"\nConfirmed: real model fetched from Hugging Face Hub, real in-browser inference, "
      f"{demo_result['latencyMs']:.0f} ms, max abs diff {demo_result['maxDiff']:.4e} "
      f"(matches the known-good weight-only int8 B/16 accuracy).")

{'adapterInfo': 'vendor=nvidia architecture=blackwell device=0x2c34', 'latencyMs': 5363.300000190735, 'maxDiff': 0.007878482341766357, 'meanDiff': 0.0012740451294125776}

Confirmed: real model fetched from Hugging Face Hub, real in-browser inference, 5363 ms, max abs diff 7.8785e-03 (matches the known-good weight-only int8 B/16 accuracy).
